# 🌧️ Nepal Disaster Dataset — Rainfall Enrichment
**Fetches daily precipitation from NASA POWER API and merges into your dataset.**

- No API key needed
- Covers all 77 Nepali districts (1982–2019)
- Run all cells top to bottom — takes 30–60 mins depending on internet speed
- Final output: `Nepal_Flood_Landslide_Enriched.csv`

## Cell 1 — Install & Import

In [ ]:
# Install required library if not already installed
!pip install requests tqdm

import pandas as pd
import numpy as np
import requests
import time
import os
from tqdm import tqdm

print('✅ All libraries loaded successfully!')

## Cell 2 — Load Your Dataset

In [ ]:
# ⚠️ UPDATE THIS PATH to wherever your CSV is saved on your computer
CSV_PATH = 'Nepal_Landslide_and_Flood_Historical_Data.csv'

df = pd.read_csv(CSV_PATH)
df['date'] = pd.to_datetime(df['date'])

print(f'✅ Dataset loaded: {df.shape[0]:,} rows, {df.shape[1]} columns')
print(f'📅 Date range: {df["date"].min().date()} → {df["date"].max().date()}')
print(f'🗺️  Unique districts: {df["district"].nunique()}')
print()
df.head()

## Cell 3 — District Coordinates Map
All 77 districts of Nepal with approximate lat/lon centroids.

In [ ]:
# GPS coordinates (lat, lon) for each Nepali district centroid
DISTRICT_COORDS = {
    'achham':            (29.0833, 81.3500),
    'arghakhanchi':      (27.9500, 83.1500),
    'baglung':           (28.2667, 83.5833),
    'baitadi':           (29.5333, 80.4167),
    'bajhang':           (29.5500, 81.1833),
    'bajura':            (29.3333, 81.5500),
    'banke':             (28.0500, 81.6167),
    'bara':              (27.0167, 85.0167),
    'bardiya':           (28.3167, 81.4833),
    'bhaktapur':         (27.6710, 85.4298),
    'bhojpur':           (27.1667, 87.0500),
    'chitwan':           (27.5291, 84.3542),
    'dadeldhura':        (29.2967, 80.5786),
    'dailekh':           (28.8458, 81.7125),
    'dang':              (28.0500, 82.3000),
    'darchula':          (29.8500, 80.5500),
    'dhading':           (27.8667, 84.9167),
    'dhankuta':          (26.9833, 87.3500),
    'dhanusa':           (26.8333, 85.9167),
    'dolakha':           (27.6667, 86.0833),
    'dolpa':             (29.0000, 82.9667),
    'doti':              (29.2667, 80.9500),
    'gorkha':            (28.3333, 84.6333),
    'gulmi':             (28.0667, 83.2667),
    'humla':             (29.9667, 81.9833),
    'ilam':              (26.9167, 87.9167),
    'jajarkot':          (28.7000, 82.1833),
    'jhapa':             (26.6333, 87.8667),
    'jumla':             (29.2833, 82.1833),
    'kailali':           (28.7167, 80.9000),
    'kalikot':           (29.1333, 81.6167),
    'kanchanpur':        (28.8500, 80.3333),
    'kapilbastu':        (27.5667, 83.0500),
    'kaski':             (28.2667, 83.9667),
    'kathmandu':         (27.7172, 85.3240),
    'kavrepalanchok':    (27.5333, 85.6833),
    'khotang':           (27.1667, 86.8333),
    'lalitpur':          (27.6588, 85.3247),
    'lamjung':           (28.2167, 84.3833),
    'mahottari':         (26.6500, 85.7833),
    'makwanpur':         (27.4333, 85.0333),
    'manang':            (28.6667, 84.0167),
    'morang':            (26.6500, 87.4500),
    'mugu':              (29.6833, 82.4333),
    'mustang':           (28.9833, 83.8500),
    'myagdi':            (28.4833, 83.5333),
    'nawalparasi east':  (27.5500, 84.3833),
    'nawalparasi west':  (27.7000, 83.7333),
    'nuwakot':           (27.9167, 85.1667),
    'okhaldhunga':       (27.3167, 86.5000),
    'palpa':             (27.8667, 83.5500),
    'panchthar':         (27.1333, 87.7833),
    'parbat':            (28.2167, 83.7000),
    'parsa':             (27.1000, 84.9833),
    'pyuthan':           (28.1000, 82.8667),
    'ramechhap':         (27.3333, 86.0833),
    'rasuwa':            (28.1500, 85.3333),
    'rautahat':          (27.0000, 85.3000),
    'rolpa':             (28.3500, 82.6500),
    'rukum east':        (28.6167, 82.6500),
    'rukum west':        (28.5500, 82.3500),
    'rupandehi':         (27.5000, 83.4500),
    'salyan':            (28.3667, 82.1667),
    'sankhuwasabha':     (27.3500, 87.3000),
    'saptari':           (26.6667, 86.7167),
    'sarlahi':           (27.0000, 85.5667),
    'sindhuli':          (27.2500, 85.9667),
    'sindhupalchok':     (27.9500, 85.6833),
    'siraha':            (26.6500, 86.2000),
    'solukhumbu':        (27.6667, 86.6667),
    'sunsari':           (26.7167, 87.1667),
    'surkhet':           (28.6000, 81.6167),
    'syangja':           (28.0833, 83.8833),
    'tanahu':            (27.9333, 84.2333),
    'taplejung':         (27.3500, 87.6667),
    'terhathum':         (27.1167, 87.5500),
    'udayapur':          (26.9167, 86.5000),
}

# Check which districts in your dataset are covered
dataset_districts = [d for d in df['district'].unique() if d != 'No Event']
covered = [d for d in dataset_districts if d in DISTRICT_COORDS]
missing = [d for d in dataset_districts if d not in DISTRICT_COORDS]

print(f'✅ Districts covered by coordinates: {len(covered)}')
if missing:
    print(f'⚠️  Districts NOT found in coords map: {missing}')
else:
    print('✅ All districts have coordinates!')

## Cell 4 — NASA POWER Fetch Function
This fetches **daily precipitation** for a district across the full date range in one API call.

In [ ]:
def fetch_nasa_rainfall(lat, lon, start_year=1982, end_year=2019, retries=3):
    """
    Fetches daily precipitation (PRECTOTCORR) from NASA POWER API.
    Returns a DataFrame with columns: [date, precipitation_mm]
    """
    url = 'https://power.larc.nasa.gov/api/temporal/daily/point'
    params = {
        'parameters': 'PRECTOTCORR',   # Precipitation corrected (mm/day)
        'community': 'RE',
        'longitude': lon,
        'latitude': lat,
        'start': f'{start_year}0101',
        'end': f'{end_year}1231',
        'format': 'JSON'
    }
    
    for attempt in range(retries):
        try:
            response = requests.get(url, params=params, timeout=60)
            if response.status_code == 200:
                data = response.json()
                rain_data = data['properties']['parameter']['PRECTOTCORR']
                # Convert to DataFrame
                rain_df = pd.DataFrame(list(rain_data.items()), columns=['date_str', 'precipitation_mm'])
                rain_df['date'] = pd.to_datetime(rain_df['date_str'], format='%Y%m%d')
                rain_df = rain_df[['date', 'precipitation_mm']]
                # NASA uses -999 for missing values
                rain_df['precipitation_mm'] = rain_df['precipitation_mm'].replace(-999.0, np.nan)
                return rain_df
            else:
                print(f'  ⚠️  HTTP {response.status_code}, retrying ({attempt+1}/{retries})...')
                time.sleep(5)
        except Exception as e:
            print(f'  ❌ Error: {e}, retrying ({attempt+1}/{retries})...')
            time.sleep(10)
    
    return None  # Return None if all retries fail

print('✅ NASA POWER fetch function defined!')
print()

# Quick test with Kathmandu
print('🔍 Testing API with Kathmandu...')
test = fetch_nasa_rainfall(27.7172, 85.3240, start_year=2019, end_year=2019)
if test is not None:
    print(f'✅ API working! Got {len(test)} days of data.')
    print(test.head())
else:
    print('❌ API test failed. Check your internet connection.')

## Cell 5 — Fetch All Districts & Save Progress
⏱️ **Expected time: 30–60 minutes** (77 districts × ~5 seconds each)

Progress is saved after every district — if it crashes, re-run this cell and it will skip already-fetched districts.

In [ ]:
CACHE_DIR = 'rainfall_cache'
os.makedirs(CACHE_DIR, exist_ok=True)

districts_to_fetch = [d for d in dataset_districts if d in DISTRICT_COORDS]
failed_districts = []

print(f'📡 Fetching rainfall for {len(districts_to_fetch)} districts...')
print(f'💾 Progress saved in: ./{CACHE_DIR}/')
print(f'⏭️  Already fetched districts will be skipped automatically.')
print()

for district in tqdm(districts_to_fetch, desc='Districts'):
    cache_file = os.path.join(CACHE_DIR, f'{district.replace(" ", "_")}.csv')
    
    # Skip if already fetched
    if os.path.exists(cache_file):
        continue
    
    lat, lon = DISTRICT_COORDS[district]
    rain_df = fetch_nasa_rainfall(lat, lon, start_year=1982, end_year=2019)
    
    if rain_df is not None:
        rain_df['district'] = district
        rain_df.to_csv(cache_file, index=False)
    else:
        failed_districts.append(district)
        print(f'\n  ⚠️  Failed: {district}')
    
    time.sleep(1)  # Be polite to the API

print()
print('✅ Fetching complete!')
if failed_districts:
    print(f'⚠️  Failed districts (will be filled with NaN): {failed_districts}')
else:
    print('🎉 All districts fetched successfully!')

## Cell 6 — Merge Rainfall Into Your Dataset

In [ ]:
# Load all cached rainfall CSVs into one big DataFrame
print('📂 Loading cached rainfall data...')
all_rain = []

for district in districts_to_fetch:
    cache_file = os.path.join(CACHE_DIR, f'{district.replace(" ", "_")}.csv')
    if os.path.exists(cache_file):
        chunk = pd.read_csv(cache_file)
        chunk['date'] = pd.to_datetime(chunk['date'])
        all_rain.append(chunk)

rainfall_df = pd.concat(all_rain, ignore_index=True)
print(f'✅ Rainfall data loaded: {len(rainfall_df):,} rows')
print()

# Compute rolling rainfall features from raw daily values
print('⚙️  Computing 3-day, 7-day, 30-day rolling sums...')
rainfall_df = rainfall_df.sort_values(['district', 'date'])
rainfall_df['rainfall_3day_sum']  = rainfall_df.groupby('district')['precipitation_mm'].transform(lambda x: x.rolling(3,  min_periods=1).sum())
rainfall_df['rainfall_7day_sum']  = rainfall_df.groupby('district')['precipitation_mm'].transform(lambda x: x.rolling(7,  min_periods=1).sum())
rainfall_df['rainfall_30day_sum'] = rainfall_df.groupby('district')['precipitation_mm'].transform(lambda x: x.rolling(30, min_periods=1).sum())
print('✅ Rolling sums computed!')
print()

# Merge into main dataset
print('🔗 Merging rainfall into main dataset...')
df_enriched = df.copy()

# Drop old empty rainfall columns
df_enriched = df_enriched.drop(columns=['precipitation', 'rainfall_3day_sum', 'rainfall_7day_sum', 'rainfall_30day_sum'])

# Merge on district + date
df_enriched = df_enriched.merge(
    rainfall_df[['district', 'date', 'precipitation_mm', 'rainfall_3day_sum', 'rainfall_7day_sum', 'rainfall_30day_sum']],
    on=['district', 'date'],
    how='left'
)

# Rename for clarity
df_enriched = df_enriched.rename(columns={'precipitation_mm': 'precipitation'})

print(f'✅ Merge complete!')
print(f'📊 Enriched dataset shape: {df_enriched.shape}')
print()
print('Missing values after merge:')
print(df_enriched[['precipitation','rainfall_3day_sum','rainfall_7day_sum','rainfall_30day_sum']].isnull().sum())
print()
df_enriched.head()

## Cell 7 — Verify & Save Final Dataset

In [ ]:
# Quick sanity check — event rows should now have real rainfall values
print('🔍 Sanity Check — Rainfall stats for EVENT rows:')
event_rows = df_enriched[df_enriched['event_occurred'] == 1]
print(event_rows[['precipitation', 'rainfall_3day_sum', 'rainfall_7day_sum']].describe())
print()

non_zero_precip = (df_enriched['precipitation'] > 0).sum()
print(f'✅ Rows with non-zero precipitation: {non_zero_precip:,} (was 0 before!)')
print()

# Save final enriched dataset
OUTPUT_PATH = 'Nepal_Flood_Landslide_Enriched.csv'
df_enriched.to_csv(OUTPUT_PATH, index=False)
print(f'💾 Final dataset saved to: {OUTPUT_PATH}')
print(f'📦 File size: {os.path.getsize(OUTPUT_PATH) / 1024 / 1024:.1f} MB')
print()
print('🎉 DONE! Your enriched dataset is ready for model training.')
print()
print('Next step → Open Nepal_Model_Training.ipynb to train your flood/landslide predictor!')

## Cell 8 — (Optional) Quick Preview of Enriched Data

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot: Average rainfall on flood days vs non-flood days
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Rainfall Distribution: Event vs No Event Days', fontsize=14, fontweight='bold')

# Flood
flood_data = df_enriched[df_enriched['district'] != 'No Event'].copy()
flood_data['Flood'] = flood_data['flood_occurred'].map({0: 'No Flood', 1: 'Flood'})
sns.boxplot(data=flood_data, x='Flood', y='rainfall_3day_sum', ax=axes[0], palette='Blues')
axes[0].set_title('3-Day Rainfall Sum vs Flood Occurrence')
axes[0].set_ylabel('Rainfall (mm)')

# Landslide
flood_data['Landslide'] = flood_data['landslide_occurred'].map({0: 'No Landslide', 1: 'Landslide'})
sns.boxplot(data=flood_data, x='Landslide', y='rainfall_3day_sum', ax=axes[1], palette='Oranges')
axes[1].set_title('3-Day Rainfall Sum vs Landslide Occurrence')
axes[1].set_ylabel('Rainfall (mm)')

plt.tight_layout()
plt.savefig('rainfall_vs_events.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Plot saved as rainfall_vs_events.png')